# Minute Forecast Lab: Monte Carlo vs Chronos-2 vs TimesFM

Three models predict where a stock's **next 1-minute price** will land, as a range (10th–90th percentile):

| Model | What it is |
|---|---|
| **Monte Carlo** | Volatility + momentum model, 10,000 simulated paths from the stock's own recent surprises |
| **Chronos-2** | Amazon's 120M-parameter time-series foundation model, fine-tuned here with LoRA |
| **TimesFM 2.5** | Google's 200M-parameter time-series foundation model, fine-tuned here with LoRA |

The notebook downloads ~4 weeks of 1-minute prices for ~45 stocks, FX pairs and crypto across 15 markets, trains the AI models,
scores all three on the last 2 trading sessions (which no model sees during training), then opens a live app that
compares them minute by minute.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all** (about 20–30 minutes).

## 1. Install and write the project files

In [ ]:
!pip install -q -U yfinance flask peft "transformers>=5" "chronos-forecasting>=2.2"
import os
os.makedirs("/content/lab/riskmc", exist_ok=True); os.makedirs("/content/lab/web", exist_ok=True)
open("/content/lab/riskmc/__init__.py", "w").close()
os.chdir("/content/lab")

In [ ]:
%%writefile /content/lab/riskmc/markets.py
"""Exchanges, trading hours and a starter ticker list (Yahoo Finance symbols).

Hours are regular sessions in local time; exchange holidays are not modeled.
"""

from __future__ import annotations

from datetime import datetime, time, timezone
from zoneinfo import ZoneInfo

MARKETS = [
    {"id": "us", "name": "United States", "exchange": "NYSE / Nasdaq", "tz": "America/New_York",
     "sessions": [("09:30", "16:00")],
     "tickers": [("AAPL", "Apple"), ("MSFT", "Microsoft"), ("NVDA", "Nvidia"),
                 ("TSLA", "Tesla"), ("JPM", "JPMorgan Chase"), ("SPY", "S&P 500 ETF")]},
    {"id": "ca", "name": "Canada", "exchange": "TSX", "tz": "America/Toronto",
     "sessions": [("09:30", "16:00")],
     "tickers": [("SHOP.TO", "Shopify"), ("RY.TO", "Royal Bank of Canada"), ("ENB.TO", "Enbridge")]},
    {"id": "br", "name": "Brazil", "exchange": "B3", "tz": "America/Sao_Paulo",
     "sessions": [("10:00", "17:00")],
     "tickers": [("PETR4.SA", "Petrobras"), ("VALE3.SA", "Vale"), ("ITUB4.SA", "Itaú Unibanco")]},
    {"id": "uk", "name": "United Kingdom", "exchange": "LSE", "tz": "Europe/London",
     "sessions": [("08:00", "16:30")],
     "tickers": [("SHEL.L", "Shell"), ("AZN.L", "AstraZeneca"), ("HSBA.L", "HSBC"), ("BP.L", "BP")]},
    {"id": "de", "name": "Germany", "exchange": "Xetra", "tz": "Europe/Berlin",
     "sessions": [("09:00", "17:30")],
     "tickers": [("SAP.DE", "SAP"), ("SIE.DE", "Siemens"), ("ALV.DE", "Allianz")]},
    {"id": "fr", "name": "France", "exchange": "Euronext Paris", "tz": "Europe/Paris",
     "sessions": [("09:00", "17:30")],
     "tickers": [("MC.PA", "LVMH"), ("TTE.PA", "TotalEnergies"), ("AIR.PA", "Airbus")]},
    {"id": "ch", "name": "Switzerland", "exchange": "SIX", "tz": "Europe/Zurich",
     "sessions": [("09:00", "17:30")],
     "tickers": [("NESN.SW", "Nestlé"), ("NOVN.SW", "Novartis"), ("ROG.SW", "Roche")]},
    {"id": "in", "name": "India", "exchange": "NSE", "tz": "Asia/Kolkata",
     "sessions": [("09:15", "15:30")],
     "tickers": [("RELIANCE.NS", "Reliance Industries"), ("TCS.NS", "Tata Consultancy"),
                 ("HDFCBANK.NS", "HDFC Bank"), ("INFY.NS", "Infosys")]},
    {"id": "sg", "name": "Singapore", "exchange": "SGX", "tz": "Asia/Singapore",
     "sessions": [("09:00", "12:00"), ("13:00", "17:00")],
     "tickers": [("D05.SI", "DBS Group"), ("O39.SI", "OCBC Bank")]},
    {"id": "hk", "name": "Hong Kong", "exchange": "HKEX", "tz": "Asia/Hong_Kong",
     "sessions": [("09:30", "12:00"), ("13:00", "16:00")],
     "tickers": [("0700.HK", "Tencent"), ("9988.HK", "Alibaba"), ("0005.HK", "HSBC Holdings")]},
    {"id": "kr", "name": "South Korea", "exchange": "KRX", "tz": "Asia/Seoul",
     "sessions": [("09:00", "15:30")],
     "tickers": [("005930.KS", "Samsung Electronics"), ("000660.KS", "SK Hynix")]},
    {"id": "jp", "name": "Japan", "exchange": "Tokyo SE", "tz": "Asia/Tokyo",
     "sessions": [("09:00", "11:30"), ("12:30", "15:30")],
     "tickers": [("7203.T", "Toyota"), ("6758.T", "Sony"), ("9984.T", "SoftBank Group")]},
    {"id": "au", "name": "Australia", "exchange": "ASX", "tz": "Australia/Sydney",
     "sessions": [("10:00", "16:00")],
     "tickers": [("BHP.AX", "BHP Group"), ("CBA.AX", "Commonwealth Bank"), ("CSL.AX", "CSL")]},
    {"id": "fx", "name": "Currencies", "exchange": "FX, 24/5", "tz": "America/New_York",
     "sessions": "fx",
     "tickers": [("EURUSD=X", "Euro / US Dollar"), ("USDJPY=X", "US Dollar / Yen"),
                 ("GBPUSD=X", "Pound / US Dollar")]},
    {"id": "crypto", "name": "Crypto", "exchange": "24/7", "tz": "UTC",
     "sessions": "always",
     "tickers": [("BTC-USD", "Bitcoin"), ("ETH-USD", "Ethereum"), ("SOL-USD", "Solana")]},
]


def _t(s: str) -> time:
    return time.fromisoformat(s)


def status(market: dict, now: datetime | None = None) -> dict:
    now = now or datetime.now(timezone.utc)
    local = now.astimezone(ZoneInfo(market["tz"]))
    sessions = market["sessions"]
    if sessions == "always":
        state = "open"
    elif sessions == "fx":
        # Sunday 17:00 to Friday 17:00 New York time
        wd, t = local.weekday(), local.time()
        closed = wd == 5 or (wd == 4 and t >= _t("17:00")) or (wd == 6 and t < _t("17:00"))
        state = "closed" if closed else "open"
    elif local.weekday() >= 5:
        state = "closed"
    else:
        t = local.time()
        if any(_t(a) <= t < _t(b) for a, b in sessions):
            state = "open"
        elif _t(sessions[0][0]) <= t < _t(sessions[-1][1]):
            state = "break"
        else:
            state = "closed"
    hours = "24/7" if sessions == "always" else "24/5" if sessions == "fx" else \
        ", ".join(f"{a}–{b}" for a, b in sessions)
    return {"state": state, "local_time": local.strftime("%a %H:%M"), "hours": hours}


def catalog() -> list[dict]:
    rows = []
    for m in MARKETS:
        rows.append({
            "id": m["id"], "name": m["name"], "exchange": m["exchange"], **status(m),
            "tickers": [{"symbol": s, "name": n} for s, n in m["tickers"]],
        })
    order = {"open": 0, "break": 1, "closed": 2}
    return sorted(rows, key=lambda r: order[r["state"]])


def market_for(symbol: str) -> dict | None:
    for m in MARKETS:
        if any(s == symbol for s, _ in m["tickers"]):
            return m
    return None

In [ ]:
%%writefile /content/lab/riskmc/minute.py
"""Next-minute Monte Carlo forecaster and the scoring shared by every model.

Model (filtered historical simulation), using only information up to the current bar:
    r_next = phi * r_last + sigma * z
- sigma: EWMA volatility (lambda = 0.97), reacts quickly to bursts
- phi:   shrunk rolling AR(1) coefficient, captures short-term momentum / bid-ask bounce
- z:     drawn from this ticker's own recent standardized forecast errors, so the
         band width and tail shape calibrate themselves per asset
The forecast distribution comes from simulating many next-minute prices.

Every model in the app reports the same nine quantiles (10th..90th percentile),
which is what TimesFM produces natively, so the comparison is like for like.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

MAX_GAP = pd.Timedelta(minutes=5)   # returns across longer gaps (overnight, lunch) are skipped
LEVELS = np.round(np.arange(0.1, 0.91, 0.1), 2)             # 0.1, 0.2, ..., 0.9
QCOLS = [f"q{int(round(l * 100))}" for l in LEVELS]          # q10 ... q90


def minute_returns(closes: pd.Series) -> pd.Series:
    """Log returns between consecutive bars; NaN where the bars are separated by a gap."""
    r = np.log(closes / closes.shift(1))
    gap = closes.index.to_series().diff() > MAX_GAP
    return r.mask(gap.to_numpy())


def model_frame(closes: pd.Series, lookback: int = 120, lam: float = 0.97,
                shrink: float = 0.5) -> pd.DataFrame:
    """Per-bar model state. Row t only uses returns up to and including bar t."""
    r = minute_returns(closes)
    sigma = np.sqrt(r.pow(2).ewm(alpha=1 - lam, adjust=False, ignore_na=True,
                                 min_periods=20).mean()).clip(lower=1e-6)
    r_prev = r.shift(1)
    num = (r * r_prev).rolling(lookback, min_periods=30).sum()
    den = (r_prev ** 2).rolling(lookback, min_periods=30).sum()
    phi = (shrink * num / den).clip(-0.3, 0.3).fillna(0.0)
    mean_next = phi * r.fillna(0.0)                       # expected return of bar t+1
    z = (r - mean_next.shift(1)) / sigma.shift(1)         # realized surprise at bar t
    return pd.DataFrame({"r": r, "sigma": sigma, "phi": phi, "mean_next": mean_next, "z": z})


def mc_quantiles(closes: pd.Series, frame: pd.DataFrame, i: int, calib: int = 300,
                 n_paths: int = 10_000, rng: np.random.Generator | None = None) -> np.ndarray | None:
    """Monte Carlo quantiles (LEVELS) of the close of bar i+1, using bars up to i."""
    rng = rng or np.random.default_rng()
    zs = frame["z"].iloc[:i + 1].dropna().to_numpy()[-calib:]
    sigma, mean = frame["sigma"].iloc[i], frame["mean_next"].iloc[i]
    if len(zs) < 60 or np.isnan(sigma):
        return None
    sim_r = mean + sigma * rng.choice(zs, n_paths)        # Monte Carlo over past surprises
    return float(closes.iloc[i]) * np.exp(np.quantile(sim_r, LEVELS))


def rescale(q: np.ndarray, scale: float) -> np.ndarray:
    """Widen (scale > 1) or narrow a quantile forecast around its median."""
    q = np.asarray(q, dtype=float)
    med = q[..., [len(LEVELS) // 2]]
    return med + scale * (q - med)


def pinball_bps(q: np.ndarray, actual: np.ndarray, last: np.ndarray) -> np.ndarray:
    """Average quantile (pinball) loss over LEVELS, in basis points of the last price.

    A proper scoring rule: it rewards ranges that are both correctly placed and tight,
    so it is the single number used to rank models.
    """
    q, actual, last = np.atleast_2d(q), np.asarray(actual)[:, None], np.asarray(last)[:, None]
    diff = actual - q
    loss = np.maximum(LEVELS * diff, (LEVELS - 1) * diff)
    return (loss / last * 1e4).mean(axis=1)


def fit_scale(q: np.ndarray, actual: np.ndarray, last: np.ndarray) -> float:
    """Spread multiplier that minimizes pinball loss on a calibration set."""
    grid = np.round(np.arange(0.4, 3.01, 0.05), 2)
    losses = [pinball_bps(rescale(q, s), actual, last).mean() for s in grid]
    return float(grid[int(np.argmin(losses))])


def score(rows: pd.DataFrame) -> dict:
    """Scores for a set of forecasts with columns q10..q90, last, actual."""
    if rows is None or rows.empty:
        return {"n": 0}
    q = rows[QCOLS].to_numpy()
    last, actual = rows["last"].to_numpy(), rows["actual"].to_numpy()
    med = rows["q50"].to_numpy()
    moved = (actual != last) & (med != last)
    direction = (np.sign(actual - last) == np.sign(med - last))[moved]
    err_model = np.abs(actual - med) / last * 1e4
    err_naive = np.abs(actual - last) / last * 1e4
    return {
        "n": int(len(rows)),
        "coverage80": float(((actual >= rows["q10"]) & (actual <= rows["q90"])).mean()),
        "width80_bps": float(((rows["q90"] - rows["q10"]) / last * 1e4).mean()),
        "pinball_bps": float(pinball_bps(q, actual, last).mean()),
        "direction_acc": float(direction.mean()) if len(direction) else None,
        "mae_bps": float(err_model.mean()),
        "naive_mae_bps": float(err_naive.mean()),
    }

In [ ]:
%%writefile /content/lab/riskmc/forecasters.py
"""The three competing next-minute forecasters behind one interface.

Each forecaster maps a list of price histories to a (batch, 9) array of price
quantiles at LEVELS (10th..90th percentile) for the next bar. A learned spread
`scale` (see minute.fit_scale) is applied on top of every model's raw output.

- Monte Carlo: filtered historical simulation (riskmc.minute)
- Chronos-2:   Amazon's time-series foundation model (chronos-forecasting, Apache 2.0)
- TimesFM 2.5: Google's time-series foundation model (Hugging Face Transformers port, Apache 2.0)
"""

from __future__ import annotations

import inspect
from pathlib import Path

import numpy as np
import pandas as pd

from .minute import LEVELS, mc_quantiles, model_frame, rescale

NAN_ROW = np.full(len(LEVELS), np.nan)


class Forecaster:
    key = "base"
    name = "Base"
    blurb = ""

    def __init__(self):
        self.scale = 1.0
        self.available = True
        self.error = None

    def raw_quantiles(self, histories: list[pd.Series]) -> np.ndarray:
        raise NotImplementedError

    def quantiles(self, histories: list[pd.Series]) -> np.ndarray:
        return rescale(self.raw_quantiles(histories), self.scale)

    def replay(self, closes: pd.Series, idxs: list[int]) -> np.ndarray:
        """Quantiles for bar i+1 for each i in idxs, using only bars up to i."""
        return self.quantiles([closes.iloc[:i + 1] for i in idxs])

    def info(self) -> dict:
        return {"key": self.key, "name": self.name, "blurb": self.blurb, "available": self.available,
                "error": self.error, "scale": self.scale, "fine_tuned": getattr(self, "fine_tuned", False)}


class MonteCarloForecaster(Forecaster):
    key, name = "mc", "Monte Carlo"
    blurb = "Volatility + momentum model, 10,000 simulated paths from the stock's own recent surprises"

    def __init__(self, n_paths: int = 10_000, seed: int | None = None):
        super().__init__()
        self.n_paths, self.rng = n_paths, np.random.default_rng(seed)

    def raw_quantiles(self, histories):
        out = []
        for h in histories:
            q = mc_quantiles(h, model_frame(h), len(h) - 1, n_paths=self.n_paths, rng=self.rng)
            out.append(NAN_ROW if q is None else q)
        return np.array(out)

    def replay(self, closes, idxs):
        frame = model_frame(closes)                       # row i only depends on bars <= i
        out = []
        for i in idxs:
            q = mc_quantiles(closes, frame, i, n_paths=self.n_paths, rng=self.rng)
            out.append(NAN_ROW if q is None else q)
        return rescale(np.array(out), self.scale)


def _batches(items, size):
    for k in range(0, len(items), size):
        yield items[k:k + size]


class Chronos2Forecaster(Forecaster):
    key, name = "chronos2", "Chronos-2"
    blurb = "Amazon's 120M-parameter time-series foundation model"

    def __init__(self, path: str = "amazon/chronos-2", device: str = "cpu", context: int = 512,
                 batch_size: int = 128):
        """`path` is a Hugging Face model id or a fine-tuned checkpoint folder (LoRA adapters load directly)."""
        super().__init__()
        from chronos import BaseChronosPipeline

        self.path, self.device, self.context, self.batch_size = path, device, context, batch_size
        self.pipe = BaseChronosPipeline.from_pretrained(path, device_map=device)
        self.fine_tuned = path != "amazon/chronos-2"

    def _predict(self, arrays: list[np.ndarray]) -> np.ndarray:
        import torch

        out = self.pipe.predict_quantiles([torch.tensor(a) for a in arrays], prediction_length=1,
                                          quantile_levels=LEVELS.tolist())
        q = out[0] if isinstance(out, tuple) else out
        if isinstance(q, (list, tuple)):                  # Chronos-2: one tensor per series
            return np.stack([np.asarray(x.float().cpu()).reshape(-1)[:len(LEVELS)] for x in q])
        return np.asarray(q.float().cpu()).reshape(len(arrays), -1)[:, :len(LEVELS)]

    def raw_quantiles(self, histories):
        arrays = [h.to_numpy(dtype=np.float32)[-self.context:] for h in histories]
        return np.concatenate([self._predict(b) for b in _batches(arrays, self.batch_size)])

    def fine_tune(self, series: list[np.ndarray], out_dir: str, num_steps: int = 600,
                  learning_rate: float = 1e-5, batch_size: int = 64, mode: str = "lora") -> None:
        """Fine-tune on minute price histories with chronos-forecasting's own `fit` (LoRA by default)."""
        if not hasattr(self.pipe, "fit"):
            raise RuntimeError("This chronos-forecasting version has no fit(); upgrade with "
                               "pip install -U chronos-forecasting")
        wanted = {"inputs": [s.astype(np.float32) for s in series], "prediction_length": 1,
                  "finetune_mode": mode, "num_steps": num_steps, "learning_rate": learning_rate, "batch_size": batch_size,
                  "context_length": self.context, "output_dir": out_dir}
        params = inspect.signature(self.pipe.fit).parameters
        kwargs = {k: v for k, v in wanted.items() if k in params}
        print("Chronos-2 fit() with", {k: v for k, v in kwargs.items() if k != "inputs"})
        self.pipe = self.pipe.fit(**kwargs)                # fit() saves to out_dir/finetuned-ckpt itself
        self.path, self.fine_tuned = str(Path(out_dir) / "finetuned-ckpt"), True


class TimesFMForecaster(Forecaster):
    key, name = "timesfm", "TimesFM 2.5"
    blurb = "Google's 200M-parameter time-series foundation model"

    def __init__(self, path: str = "google/timesfm-2.5-200m-transformers", device: str = "cpu",
                 context: int = 512, batch_size: int = 128, adapter: str | None = None):
        """`adapter` is a folder with LoRA weights saved by `fine_tune`, applied on top of the base model."""
        super().__init__()
        from transformers import TimesFm2_5ModelForPrediction

        self.device, self.context, self.batch_size = device, context, batch_size
        self.model = TimesFm2_5ModelForPrediction.from_pretrained(path, torch_dtype="float32").to(device)
        self.fine_tuned = adapter is not None
        if adapter:
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, adapter).to(device)
        self.model.eval()

    def raw_quantiles(self, histories):
        import torch

        arrays = [h.to_numpy(dtype=np.float32)[-self.context:] for h in histories]
        out = []
        with torch.no_grad():
            for b in _batches(arrays, self.batch_size):
                res = self.model(past_values=[torch.tensor(a, device=self.device) for a in b],
                                 forecast_context_len=self.context)
                # full_predictions: (batch, horizon, 10) = mean, then the 10th..90th percentiles
                out.append(res.full_predictions[:, 0, 1:1 + len(LEVELS)].float().cpu().numpy())
        return np.concatenate(out)

    def fine_tune(self, series: list[np.ndarray], out_dir: str, num_steps: int = 600,
                  learning_rate: float = 1e-4, batch_size: int = 32, lora_r: int = 4, seed: int = 0) -> list[float]:
        """LoRA fine-tuning, following TimesFM's own example (timesfm-forecasting/examples/finetuning).

        Each step samples random (512 context -> next price) windows from the training
        series; the model's built-in loss (MSE + quantile loss in normalized space) is minimized.
        """
        import torch
        from peft import LoraConfig, get_peft_model

        rng = np.random.default_rng(seed)
        series = [s for s in series if len(s) > self.context + 1]
        model = get_peft_model(self.model, LoraConfig(r=lora_r, lora_alpha=2 * lora_r, target_modules="all-linear",
                                                      lora_dropout=0.05, bias="none"))
        model.print_trainable_parameters()
        opt = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_steps)
        losses = []
        model.train()
        for step in range(1, num_steps + 1):
            ctx, tgt = [], []
            for _ in range(batch_size):
                s = series[rng.integers(len(series))]
                start = rng.integers(0, len(s) - self.context - 1)
                ctx.append(torch.tensor(s[start:start + self.context], device=self.device))
                tgt.append(float(s[start + self.context]))
            loss = model(past_values=ctx, future_values=torch.tensor(tgt, device=self.device)[:, None],
                         forecast_context_len=self.context).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(); sched.step()
            losses.append(loss.item())
            if step % 50 == 0 or step == num_steps:
                print(f"  TimesFM step {step}/{num_steps}  loss {np.mean(losses[-50:]):.4f}")
        model.eval()
        model.save_pretrained(out_dir)
        self.model, self.fine_tuned = model, True
        return losses


def build(load_ai: bool = True, device: str | None = None, chronos_path: str | None = None,
          timesfm_adapter: str | None = None, scales: dict | None = None) -> dict[str, Forecaster]:
    """All forecasters that can be loaded here. A model that fails to load stays listed as unavailable."""
    if device is None:
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            device = "cpu"
    models: dict[str, Forecaster] = {"mc": MonteCarloForecaster()}
    specs = [("chronos2", lambda: Chronos2Forecaster(chronos_path or "amazon/chronos-2", device)),
             ("timesfm", lambda: TimesFMForecaster(device=device, adapter=timesfm_adapter))]
    for key, make in specs:
        if not load_ai:
            continue
        try:
            models[key] = make()
        except Exception as e:                            # keep the app usable without this model
            stub = Forecaster()
            stub.key = key
            stub.name = {"chronos2": Chronos2Forecaster.name, "timesfm": TimesFMForecaster.name}[key]
            stub.available, stub.error = False, f"{type(e).__name__}: {e}"
            models[key] = stub
            print(f"Could not load {stub.name}: {stub.error}")
    for key, s in (scales or {}).items():
        if key in models:
            models[key].scale = float(s)
    return models

In [ ]:
%%writefile /content/lab/riskmc/training.py
"""Download minute history, train each forecaster, and score them on unseen days.

Time is split three ways per ticker, so nothing is scored on data it learned from:
    [ fine-tune ............ ][ calibrate: 1 session ][ test: last 2 sessions ]
- fine-tune: Chronos-2 is fine-tuned on these prices
- calibrate: each model's range width (spread scale) is fitted here
- test:      all three models are scored here, out of sample
"""

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .markets import MARKETS
from .minute import LEVELS, MAX_GAP, QCOLS, fit_scale, score

TRAIN_TICKERS = [s for m in MARKETS for s, _ in m["tickers"]]


def download_minutes(symbol: str, days: int = 28, retries: int = 3) -> pd.Series:
    """Completed 1-minute closes for the last `days` days (Yahoo serves 7 days per request)."""
    import yfinance as yf

    end = pd.Timestamp.now(tz="UTC").normalize() + pd.Timedelta(days=1)
    parts = []
    for k in range(int(np.ceil(days / 7))):
        e, s = end - pd.Timedelta(days=7 * k), end - pd.Timedelta(days=7 * (k + 1))
        for attempt in range(retries):
            try:
                h = yf.Ticker(symbol).history(start=s.strftime("%Y-%m-%d"), end=e.strftime("%Y-%m-%d"),
                                              interval="1m", auto_adjust=True)
                break
            except Exception:
                time.sleep(2 * (attempt + 1))
        else:
            continue
        if not h.empty:
            parts.append(h["Close"])
    if not parts:
        return pd.Series(dtype=float)
    closes = pd.concat(parts).sort_index()
    closes = closes[~closes.index.duplicated()].dropna()
    closes = closes[closes.index.second == 0]
    return closes[closes.index + pd.Timedelta(minutes=1) <= pd.Timestamp.now(tz=closes.index.tz)]


def download_all(tickers=TRAIN_TICKERS, days: int = 28, min_bars: int = 2_000) -> dict[str, pd.Series]:
    data = {}
    for i, t in enumerate(tickers, 1):
        s = download_minutes(t, days)
        status = f"{len(s):,} bars" if len(s) >= min_bars else f"skipped ({len(s)} bars)"
        print(f"[{i}/{len(tickers)}] {t:<12} {status}")
        if len(s) >= min_bars:
            data[t] = s
    return data


def split_points(closes: pd.Series, test_sessions: int = 2, calib_sessions: int = 1):
    """Timestamps where the calibration and test windows start (by trading session)."""
    sessions = pd.Index(closes.index.normalize().unique())
    if len(sessions) < test_sessions + calib_sessions + 3:
        return None
    test_start = sessions[-test_sessions]
    calib_start = sessions[-(test_sessions + calib_sessions)]
    return calib_start, test_start


def eval_indices(closes: pd.Series, start, end, n: int, min_history: int = 600) -> list[int]:
    """Up to n evenly spaced bars i whose next bar (the target) falls in [start, end)."""
    t = closes.index
    ok = [i for i in range(min_history, len(closes) - 1)
          if start <= t[i + 1] < end and t[i + 1] - t[i] <= MAX_GAP]
    if len(ok) > n:
        ok = [ok[k] for k in np.linspace(0, len(ok) - 1, n).astype(int)]
    return ok


def collect(models: dict, data: dict[str, pd.Series], window: str, n_per_ticker: int) -> pd.DataFrame:
    """Forecast every sampled point in the given window with every model."""
    rows = []
    for sym, closes in data.items():
        sp = split_points(closes)
        if sp is None:
            continue
        calib_start, test_start = sp
        start, end = (calib_start, test_start) if window == "calib" else (test_start, closes.index[-1] + MAX_GAP)
        idxs = eval_indices(closes, start, end, n_per_ticker)
        if not idxs:
            continue
        last = closes.to_numpy()[idxs]
        actual = closes.to_numpy()[[i + 1 for i in idxs]]
        for key, m in models.items():
            if not m.available:
                continue
            q = m.replay(closes, idxs)
            df = pd.DataFrame(q, columns=QCOLS)
            df["model"], df["symbol"], df["last"], df["actual"] = key, sym, last, actual
            df["time"] = closes.index[[i + 1 for i in idxs]]
            rows.append(df.dropna(subset=QCOLS))
        print(f"  {window}: {sym:<12} {len(idxs)} points")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def calibrate(models: dict, calib_rows: pd.DataFrame) -> dict[str, float]:
    """Fit each model's spread scale on the calibration window (raw outputs, scale = 1)."""
    scales = {}
    for key, m in models.items():
        r = calib_rows[calib_rows["model"] == key]
        if r.empty:
            continue
        m.scale = fit_scale(r[QCOLS].to_numpy(), r["actual"].to_numpy(), r["last"].to_numpy())
        scales[key] = m.scale
        print(f"  {m.name:<12} spread scale = {m.scale:.2f}")
    return scales


def summarize(models: dict, rows: pd.DataFrame) -> pd.DataFrame:
    out = []
    for key, m in models.items():
        r = rows[rows["model"] == key]
        if r.empty:
            continue
        s = score(r)
        out.append({"model": m.name, "points": s["n"], "in 8-in-10 range": s["coverage80"],
                    "avg range width (bps)": s["width80_bps"], "pinball loss (bps)": s["pinball_bps"],
                    "direction right": s["direction_acc"]})
    return pd.DataFrame(out).set_index("model").sort_values("pinball loss (bps)")


def save_report(path: str | Path, models: dict, zero_shot: pd.DataFrame, trained: pd.DataFrame,
                scales: dict, data: dict, extra: dict | None = None) -> dict:
    report = {
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="minutes"),
        "tickers": len(data),
        "bars": int(sum(len(s) for s in data.values())),
        "scales": scales,
        "zero_shot": zero_shot.reset_index().to_dict(orient="records"),
        "trained": trained.reset_index().to_dict(orient="records"),
        **(extra or {}),
    }
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(report, indent=2, default=str))
    return report


def finetune_series(data: dict[str, pd.Series]) -> list[np.ndarray]:
    """Per-ticker price histories that end before the calibration window."""
    out = []
    for closes in data.values():
        sp = split_points(closes)
        if sp is not None:
            out.append(closes[closes.index < sp[0]].to_numpy(dtype=np.float32))
    return out

In [ ]:
%%writefile /content/lab/app.py
"""Minute Forecast Lab: Monte Carlo vs Chronos-2 vs TimesFM, scored live on real minute prices.

    python app.py              # local, Monte Carlo only (AI models load if installed)
    In Colab: see colab/Minute_Forecast_Lab.ipynb, which calls serve(models, report)
"""

from __future__ import annotations

import json
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from flask import Flask, jsonify, send_from_directory

from riskmc.markets import catalog, market_for, status
from riskmc.minute import MAX_GAP, QCOLS, score

ROOT = Path(__file__).resolve().parent
app = Flask(__name__, static_folder=str(ROOT / "web"))

CACHE_SECONDS = 5
REPLAY_BARS = 120
MODELS: dict = {}
REPORT: dict | None = None

_lock = threading.Lock()
_model_lock = threading.Lock()                               # one model call at a time (shared GPU)
_bars: dict[str, tuple[float, pd.Series, dict]] = {}       # symbol -> (fetched_at, closes, meta)
_replays: dict[str, tuple[pd.Timestamp, pd.DataFrame]] = {}
_live: dict[str, dict[pd.Timestamp, dict]] = {}           # symbol -> base bar -> {made_at, last, q: {model: array}}
_feed: dict[str, dict] = {}                               # symbol -> newest bar seen + feed latency


def get_bars(symbol: str) -> tuple[pd.Series, dict]:
    """Completed 1-minute closes for the last few sessions (cached briefly)."""
    with _lock:
        hit = _bars.get(symbol)
    if hit and time.time() - hit[0] < CACHE_SECONDS:
        return hit[1], hit[2]

    t = yf.Ticker(symbol)
    hist = t.history(period="5d", interval="1m", auto_adjust=True)
    if hist.empty:
        raise LookupError(f"No minute data for {symbol}")
    closes = hist["Close"].dropna()
    closes = closes[closes.index.second == 0]                          # drop Yahoo's partial "live" row
    now = pd.Timestamp.now(tz=closes.index.tz)
    closes = closes[closes.index + pd.Timedelta(minutes=1) <= now]     # drop the bar still forming
    meta = t.history_metadata or {}
    with _lock:
        _bars[symbol] = (time.time(), closes, meta)
    return closes, meta


def available_models() -> dict:
    return {k: m for k, m in MODELS.items() if m.available}


def get_replay(symbol: str, closes: pd.Series) -> pd.DataFrame:
    """Every model's forecast for each of the last REPLAY_BARS bars, using only earlier bars."""
    last = closes.index[-1]
    with _lock:
        hit = _replays.get(symbol)
    if hit and hit[0] == last:
        return hit[1]
    t = closes.index
    idxs = [i for i in range(max(600, len(closes) - REPLAY_BARS - 1), len(closes) - 1)
            if t[i + 1] - t[i] <= MAX_GAP]
    frames = []
    if idxs:
        for key, m in available_models().items():
            with _model_lock:
                q = m.replay(closes, idxs)
            df = pd.DataFrame(q, columns=QCOLS)
            df["model"], df["time"] = key, t[[i + 1 for i in idxs]]
            df["last"], df["actual"] = closes.to_numpy()[idxs], closes.to_numpy()[[i + 1 for i in idxs]]
            frames.append(df.dropna(subset=QCOLS))
    rp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    with _lock:
        _replays[symbol] = (last, rp)
    return rp


def update_live(symbol: str, closes: pd.Series) -> tuple[dict, list[dict]]:
    """Lock in every model's forecast the first time a bar is seen; score it when the next bar lands."""
    base = closes.index[-1]
    with _lock:
        have = base in _live.get(symbol, {})
    if not have:
        q = {}
        for key, m in available_models().items():
            with _model_lock:
                row = m.quantiles([closes])[0]
            if np.all(np.isfinite(row)):
                q[key] = row
        with _lock:
            _live.setdefault(symbol, {})[base] = {
                "made_at": datetime.now(timezone.utc), "last": float(closes.iloc[-1]), "q": q}

    results = []
    with _lock:
        preds = dict(_live[symbol])
    for b, p in sorted(preds.items()):
        later = closes[closes.index > b]
        if later.empty:
            continue
        nxt = later.index[0]
        results.append({"base": b, "time": nxt, "last": p["last"], "actual": float(later.iloc[0]),
                        "gap": nxt - b > MAX_GAP, "q": p["q"]})
    return preds[base], results


def feed_latency(symbol: str, closes: pd.Series) -> float:
    """Seconds between a bar closing and Yahoo publishing it, learned as new bars appear."""
    newest = closes.index[-1]
    since_close = (pd.Timestamp.now(tz=newest.tz) - newest).total_seconds() - 60
    with _lock:
        feed = _feed.get(symbol)
        if feed is None:
            # First look: a fresh feed is assumed ~20s behind; a stale one is at least this far behind.
            feed = _feed[symbol] = {"bar": newest, "latency": since_close if since_close > 120 else 20.0}
        elif newest > feed["bar"]:
            if newest - feed["bar"] <= pd.Timedelta(minutes=2):      # we watched it arrive
                feed["latency"] = 0.5 * feed["latency"] + 0.5 * max(since_close, 0.0)
            feed["bar"] = newest
        return feed["latency"]


def _hhmm(ts: pd.Timestamp) -> str:
    return ts.strftime("%H:%M")


def _clean(x):
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    return x


def _qdict(q) -> dict:
    return {c: float(v) for c, v in zip(QCOLS, q)}


def _inside(q, actual) -> bool:
    return bool(q[0] <= actual <= q[-1])


@app.get("/")
def index():
    return send_from_directory(app.static_folder, "index.html")


@app.get("/api/markets")
def markets():
    return jsonify({"markets": catalog(), "utc": datetime.now(timezone.utc).isoformat()})


@app.get("/api/forecast/<symbol>")
def forecast_api(symbol: str):
    symbol = symbol.upper().strip()
    try:
        closes, meta = get_bars(symbol)
    except Exception as e:
        return jsonify({"error": f"Couldn't load {symbol}: {e}"}), 404
    if len(closes) < 700:
        return jsonify({"error": f"Not enough minute history for {symbol} yet (need about 2 sessions)"}), 404

    replay = get_replay(symbol, closes)
    pred, results = update_live(symbol, closes)
    keys = list(available_models())

    m = market_for(symbol)
    latency = feed_latency(symbol, closes)
    target = closes.index[-1] + pd.Timedelta(minutes=1)
    expected_in = (target + pd.Timedelta(minutes=1, seconds=latency)
                   - pd.Timestamp.now(tz=target.tz)).total_seconds()
    age = (pd.Timestamp.now(tz=closes.index.tz) - closes.index[-1]).total_seconds() - 60

    # chart: last hour of actual prices with each model's replayed 8-in-10 range
    by_model = {k: replay[replay["model"] == k].set_index("time") for k in keys} if not replay.empty else {}
    points = []
    for t, c in closes.iloc[-60:].items():
        p = {"t": _hhmm(t), "actual": float(c), "ranges": {}}
        for k, df in by_model.items():
            if t in df.index:
                p["ranges"][k] = [float(df.at[t, "q10"]), float(df.at[t, "q90"])]
        points.append(p)

    # live: every scored forecast this session, per model
    scored = [r for r in results if not r["gap"]]
    live_scores = {}
    for k in keys:
        rows = pd.DataFrame([{**_qdict(r["q"][k]), "last": r["last"], "actual": r["actual"]}
                             for r in scored if k in r["q"]])
        live_scores[k] = score(rows)
    last_result = None
    if scored:
        r = scored[-1]
        last_result = {"target": _hhmm(r["time"]), "actual": r["actual"], "models": {
            k: {"q10": float(r["q"][k][0]), "q90": float(r["q"][k][-1]), "inside": _inside(r["q"][k], r["actual"])}
            for k in keys if k in r["q"]}}
    streak = {k: [{"t": _hhmm(r["time"]), "inside": _inside(r["q"][k], r["actual"])}
                  for r in scored[-40:] if k in r["q"]] for k in keys}
    log = [{"target": _hhmm(r["time"]), "actual": r["actual"], "gap": bool(r["gap"]),
            "models": {k: [float(r["q"][k][0]), float(r["q"][k][-1])] for k in keys if k in r["q"]}}
           for r in reversed(results[-30:])]

    day_open = closes[closes.index.normalize() == closes.index[-1].normalize()].iloc[0]
    return jsonify(_clean({
        "symbol": symbol,
        "name": meta.get("longName") or meta.get("shortName") or symbol,
        "exchange": m["exchange"] if m else meta.get("exchangeName", ""),
        "currency": meta.get("currency", ""),
        "tz": str(closes.index.tz),
        "market": status(m) if m else None,
        "last_price": float(closes.iloc[-1]),
        "last_bar": _hhmm(closes.index[-1]),
        "session_change": float(closes.iloc[-1] / day_open - 1),
        "data_age_sec": max(0.0, age),
        "models": [MODELS[k].info() for k in MODELS],
        "report": REPORT,
        "forecast": {"target": _hhmm(target), "last": pred["last"],
                     "models": {k: _qdict(q) for k, q in pred["q"].items()}},
        "points": points,
        "replay": {k: score(replay[replay["model"] == k]) if not replay.empty else {"n": 0} for k in keys},
        "live": {
            "made_at_local": pred["made_at"].astimezone(closes.index.tz).strftime("%H:%M:%S"),
            "made_at": pred["made_at"].isoformat(),
            "expected_in_sec": expected_in,
            "latency_sec": latency,
            "last_result": last_result,
            "streak": streak,
            "scores": live_scores,
            "log": log,
        },
    }))


def serve(models: dict, report: dict | None = None, port: int = 5050, host: str = "127.0.0.1",
          background: bool = False):
    """Run the app with the given forecasters (the Colab notebook runs it in the background)."""
    global MODELS, REPORT
    MODELS, REPORT = models, report
    run = lambda: app.run(host=host, port=port, threaded=True, use_reloader=False)
    if background:
        th = threading.Thread(target=run, daemon=True)
        th.start()
        return th
    run()


if __name__ == "__main__":
    import argparse

    from riskmc.forecasters import build

    p = argparse.ArgumentParser()
    p.add_argument("--no-ai", action="store_true", help="Monte Carlo only")
    p.add_argument("--models-dir", default=str(ROOT / "models"))
    a = p.parse_args()
    report_path = Path(a.models_dir) / "report.json"
    report = json.loads(report_path.read_text()) if report_path.exists() else None
    chronos_dir, timesfm_dir = Path(a.models_dir) / "chronos2-lora", Path(a.models_dir) / "timesfm-lora"
    models = build(load_ai=not a.no_ai,
                   chronos_path=str(chronos_dir) if chronos_dir.exists() else None,
                   timesfm_adapter=str(timesfm_dir) if timesfm_dir.exists() else None,
                   scales=(report or {}).get("scales") if report else None)
    for m in models.values():
        state = "not loaded" if not m.available else "trained" if m.info()["fine_tuned"] else "base model"
        print(f"  {m.name:<12} {state}, range width x{m.scale:.2f}")
    print("Minute Forecast Lab → http://127.0.0.1:5050")
    serve(models, report)

In [ ]:
%%writefile /content/lab/web/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Minute Forecast Lab</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
/* Layout: one centered column. Forecasts → live check → scoreboard → chart → training details. */
:root {
  --bg: #f4f6f5; --panel: #ffffff; --ink: #16201d; --muted: #5d6b66; --faint: #8a9792; --line: #e0e5e2; --track: #eef1ef;
  --accent: #16201d;
  --s-mc: #2a78d6; --s-chronos2: #eb6834; --s-timesfm: #1baf7a;
  --good: #1a7f4b; --good-soft: #dcf1e5; --bad: #c0392b; --bad-soft: #f8e1de; --warn: #a86a00; --warn-soft: #f6ead2;
  --sans: "IBM Plex Sans", system-ui, -apple-system, "Segoe UI", sans-serif;
  --mono: "IBM Plex Mono", ui-monospace, "SF Mono", Menlo, monospace;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --bg: #0f1413; --panel: #161d1b; --ink: #e6ecea; --muted: #97a6a1; --faint: #6c7a75; --line: #26302d; --track: #1d2523;
    --accent: #e6ecea;
    --s-mc: #3987e5; --s-chronos2: #d95926; --s-timesfm: #199e70;
    --good: #46c083; --good-soft: #163325; --bad: #f07167; --bad-soft: #3a1d1a; --warn: #e2a93b; --warn-soft: #3a2d14;
    color-scheme: dark;
  }
}
:root[data-theme="dark"] {
  --bg: #0f1413; --panel: #161d1b; --ink: #e6ecea; --muted: #97a6a1; --faint: #6c7a75; --line: #26302d; --track: #1d2523;
  --accent: #e6ecea;
  --s-mc: #3987e5; --s-chronos2: #d95926; --s-timesfm: #199e70;
  --good: #46c083; --good-soft: #163325; --bad: #f07167; --bad-soft: #3a1d1a; --warn: #e2a93b; --warn-soft: #3a2d14;
  color-scheme: dark;
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink); font: 15px/1.5 var(--sans); }
.num { font-family: var(--mono); font-variant-numeric: tabular-nums; }
.wrap { max-width: 960px; margin: 0 auto; padding-block: 20px 48px;
  padding-left: max(16px, env(safe-area-inset-left, 0px)); padding-right: max(16px, env(safe-area-inset-right, 0px)); }

.top { display: flex; flex-wrap: wrap; align-items: center; gap: 10px 16px; margin-bottom: 20px; }
.brand { font-weight: 600; margin-right: auto; }
select, input, button { font: inherit; color: var(--ink); background: var(--panel); border: 1px solid var(--line);
  border-radius: 8px; padding: 8px 12px; }
select { min-width: 0; max-width: 100%; cursor: pointer; }
input { width: 170px; }
button { cursor: pointer; }
:focus-visible { outline: 2px solid var(--s-mc); outline-offset: 2px; }
.picker { display: flex; flex-wrap: wrap; gap: 8px; min-width: 0; }
.custom { display: flex; gap: 6px; }

.card { background: var(--panel); border: 1px solid var(--line); border-radius: 14px; padding: 22px 24px; min-width: 0; }
.stack { display: flex; flex-direction: column; gap: 16px; }
.eyebrow { font-size: 12px; letter-spacing: .07em; text-transform: uppercase; color: var(--faint); font-weight: 600; }
.sub { color: var(--muted); font-size: 13px; }
h2 { margin: 0; font-size: 16px; font-weight: 600; }
.card-head { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: 8px 16px; margin-bottom: 14px; }

.id-row { display: flex; flex-wrap: wrap; justify-content: space-between; gap: 8px 20px; align-items: baseline; }
.id-row h1 { margin: 0; font-size: 20px; font-weight: 600; text-wrap: balance; }
.price { font-size: 20px; font-weight: 500; text-align: right; }
.chip { display: inline-flex; align-items: center; gap: 5px; font-size: 12px; font-weight: 500; padding: 2px 9px; border-radius: 99px; white-space: nowrap; }
.chip.open, .chip.inside { color: var(--good); background: var(--good-soft); }
.chip.delayed { color: var(--warn); background: var(--warn-soft); }
.chip.closed, .chip.na { color: var(--muted); background: var(--track); }
.chip.outside { color: var(--bad); background: var(--bad-soft); }

/* Model identity */
.sw { display: inline-block; width: 10px; height: 10px; border-radius: 3px; flex: none; }
.mname { display: inline-flex; align-items: center; gap: 8px; font-weight: 500; white-space: nowrap; }

/* Forecast rows on one shared axis */
.fc { margin-top: 22px; }
.fc-rows { display: grid; grid-template-columns: max-content max-content 1fr; gap: 12px 18px; align-items: center; margin-top: 12px; position: relative; }
.fc-range { font-size: 14px; }
.fc-na { grid-column: 2 / -1; color: var(--muted); font-size: 13px; }
.lane { position: relative; height: 22px; }
.lane .track { position: absolute; inset: 10px 0 auto; height: 2px; background: var(--line); }
.lane .band { position: absolute; top: 3px; height: 16px; border-radius: 4px; opacity: .85; }
.lane .med { position: absolute; top: 0; width: 2px; height: 22px; background: var(--panel); }
.lane .now { position: absolute; top: -6px; bottom: -6px; width: 0; border-left: 2px dashed var(--ink); opacity: .7; }
.axis-row { grid-column: 3; display: flex; justify-content: space-between; font-size: 12px; color: var(--faint); }
.fc-note { margin-top: 14px; color: var(--muted); font-size: 13px; }

/* Live check */
.live-dot { width: 8px; height: 8px; border-radius: 50%; background: var(--good); display: inline-block; margin-right: 8px; animation: pulse 1.6s ease-in-out infinite; }
.live-dot.off { background: var(--faint); animation: none; }
@keyframes pulse { 50% { opacity: .25; } }
.steps { display: grid; grid-template-columns: 1fr 1fr 1.3fr; gap: 12px; }
.step { border: 1px solid var(--line); border-radius: 10px; padding: 12px 14px; min-width: 0; transition: background .6s, border-color .6s; }
.step.active { border-color: var(--ink); }
.step-n { display: inline-grid; place-items: center; width: 20px; height: 20px; border-radius: 50%; font-size: 12px; font-weight: 600;
  background: var(--track); color: var(--muted); margin-right: 8px; flex: none; }
.step.active .step-n { background: var(--ink); color: var(--panel); }
.step-title { font-size: 13px; color: var(--muted); display: flex; align-items: center; }
.step-main { font-size: 17px; font-weight: 500; margin-top: 6px; }
.step-sub { font-size: 12px; color: var(--muted); margin-top: 2px; }
.progress { height: 6px; border-radius: 99px; background: var(--track); margin-top: 10px; overflow: hidden; }
.progress > div { height: 100%; background: var(--ink); border-radius: 99px; transition: width 1s linear; }
.step.flash { background: var(--good-soft); border-color: var(--good); }
.res { display: flex; flex-direction: column; gap: 4px; margin-top: 8px; font-size: 13px; }
.res > div { display: flex; justify-content: space-between; align-items: center; gap: 8px; }
.streaks { display: grid; grid-template-columns: max-content 1fr max-content; gap: 8px 14px; align-items: center; margin-top: 16px; padding-top: 14px;
  border-top: 1px solid var(--line); font-size: 13px; }
.cells { display: flex; flex-wrap: wrap; gap: 3px; min-width: 0; }
.cell { width: 12px; height: 12px; border-radius: 3px; background: var(--good); }
.cell.miss { background: var(--bad); }
.cell.empty { background: transparent; border: 1px dashed var(--line); }
.tally { color: var(--muted); white-space: nowrap; }

/* Scoreboard */
.seg { display: inline-flex; background: var(--track); border-radius: 9px; padding: 3px; }
.seg button { border: 0; background: transparent; padding: 5px 12px; border-radius: 7px; font-size: 13px; color: var(--muted); }
.seg button[aria-pressed="true"] { background: var(--panel); color: var(--ink); font-weight: 500; box-shadow: 0 1px 2px rgba(0,0,0,.08); }
.table-wrap { overflow-x: auto; }
table { width: 100%; border-collapse: collapse; font-size: 14px; }
th, td { padding: 10px 12px; text-align: right; border-bottom: 1px solid var(--line); white-space: nowrap; }
th { color: var(--faint); font-weight: 600; font-size: 12px; vertical-align: bottom; }
th small { display: block; font-weight: 400; color: var(--faint); }
th:first-child, td:first-child { text-align: left; }
tr:last-child td { border-bottom: 0; }
.best { font-size: 11px; font-weight: 600; color: var(--good); background: var(--good-soft); border-radius: 99px; padding: 1px 7px; margin-left: 6px; }
.cov-good { color: var(--good); } .cov-warn { color: var(--warn); } .cov-bad { color: var(--bad); }
.note { color: var(--muted); font-size: 13px; margin-top: 12px; max-width: 75ch; }

/* Chart */
.toggles { display: flex; flex-wrap: wrap; gap: 6px; }
.toggle { display: inline-flex; align-items: center; gap: 7px; font-size: 13px; padding: 4px 10px; border-radius: 99px; }
.toggle[aria-pressed="false"] { opacity: .45; }
.toggle[aria-pressed="false"] .sw { background: transparent !important; outline: 1.5px solid var(--faint); outline-offset: -1.5px; }
.chart-box { position: relative; height: 320px; }
.legend-line { display: flex; flex-wrap: wrap; gap: 14px; font-size: 12px; color: var(--muted); margin-top: 8px; }
.dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; background: var(--ink); margin-right: 6px; vertical-align: middle; }

details { border-top: 1px solid var(--line); padding: 12px 0; }
details:first-of-type { border-top: 0; }
summary { cursor: pointer; font-weight: 500; }
details p { color: var(--muted); max-width: 72ch; margin: 10px 0; }
details table { font-size: 13px; margin-top: 8px; }

.state { text-align: center; padding: 72px 16px; color: var(--muted); }
.state h2 { color: var(--ink); font-size: 20px; margin: 0 0 6px; }
.err { color: var(--bad); }
.updated { color: var(--faint); font-size: 12px; }

@media (max-width: 720px) {
  .card { padding: 18px; }
  .steps { grid-template-columns: 1fr; }
  .fc-rows { grid-template-columns: max-content 1fr; }
  .fc-rows .lane, .axis-row { grid-column: 1 / -1; }
  .fc-range { text-align: right; }
  .price { text-align: left; }
  .chart-box { height: 260px; }
  .picker, .picker select { width: 100%; }
  .custom { width: 100%; } .custom input { flex: 1; }
  .streaks { grid-template-columns: max-content 1fr; }
  .streaks .cells { grid-column: 1 / -1; grid-row: auto; }
}
@media (prefers-reduced-motion: reduce) { * { transition: none !important; animation: none !important; } }
</style>
</head>
<body>
<div class="wrap">
  <div class="top">
    <div class="brand">Minute Forecast Lab</div>
    <div class="picker">
      <select id="ticker" aria-label="Choose a ticker"><option>Loading markets…</option></select>
      <form class="custom" id="custom">
        <input id="symbol" placeholder="Other symbol" aria-label="Any Yahoo Finance symbol" autocomplete="off">
        <button type="submit">Go</button>
      </form>
    </div>
  </div>
  <main id="main" class="stack">
    <div class="card state"><h2>Pick a ticker</h2>Markets that are open right now are listed first.</div>
  </main>
</div>

<script>
const POLL_MS = 10000;
let current = null, pollTimer = null, chart = null, lastUpdate = 0, liveClock = null;
let lastLiveTarget = null, flashUntil = 0;
let tab = 'replay';
const hidden = new Set();
try { tab = localStorage.getItem('mfl-tab') || 'replay'; JSON.parse(localStorage.getItem('mfl-hidden') || '[]').forEach(k => hidden.add(k)); } catch {}

const tok = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const color = k => tok('--s-' + k) || tok('--muted');
const esc = s => String(s ?? '').replace(/[&<>"]/g, c => ({'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]));
const pct = x => x == null ? '–' : Math.round(x * 100) + '%';
let digits = 2;
const fmt = x => x == null ? '–' : x.toLocaleString(undefined, { minimumFractionDigits: digits, maximumFractionDigits: digits });
const signed = (x, d = 2) => (x > 0 ? '+' : x < 0 ? '−' : '') + Math.abs(x).toFixed(d);
const sw = k => `<span class="sw" style="background:var(--s-${esc(k)})"></span>`;
const mname = m => `<span class="mname">${sw(m.key)}${esc(m.name)}</span>`;

/* ---------- picker ---------- */
async function loadMarkets() {
  const { markets } = await fetch('api/markets').then(r => r.json());
  const label = { open: 'open now', break: 'lunch break', closed: 'closed' };
  const sel = document.getElementById('ticker');
  sel.innerHTML = '<option value="">Choose a ticker…</option>' + markets.map(m =>
    `<optgroup label="${esc(m.name)} · ${esc(m.exchange)} · ${label[m.state]}">${m.tickers.map(t =>
      `<option value="${esc(t.symbol)}">${esc(t.name)} (${esc(t.symbol)})</option>`).join('')}</optgroup>`).join('');
  if (current) {
    if (![...sel.options].some(o => o.value === current)) sel.add(new Option(current, current));
    sel.value = current;
  }
  return markets;
}
document.getElementById('ticker').addEventListener('change', e => e.target.value && select(e.target.value));
document.getElementById('custom').addEventListener('submit', e => {
  e.preventDefault();
  const v = document.getElementById('symbol').value.trim().toUpperCase();
  if (v) { select(v); document.getElementById('symbol').value = ''; }
});

function select(sym) {
  current = sym; lastUpdate = 0; lastLiveTarget = null;
  if (chart) { chart.destroy(); chart = null; }
  try { localStorage.setItem('mfl-ticker', sym); } catch {}
  const sel = document.getElementById('ticker');
  if (![...sel.options].some(o => o.value === sym)) sel.add(new Option(sym, sym));
  sel.value = sym;
  document.getElementById('main').innerHTML = `<div class="card state"><h2>Loading ${esc(sym)}</h2>Fetching minute prices and replaying the last two hours with every model…</div>`;
  clearTimeout(pollTimer); clearInterval(liveClock); poll();
}

function schedulePoll(ms) { clearTimeout(pollTimer); pollTimer = setTimeout(poll, ms); }

async function poll() {
  const sym = current;
  let d;
  try { d = await fetch('api/forecast/' + encodeURIComponent(sym)).then(r => r.json()); }
  catch { d = { error: 'Lost connection to the server. Retrying every 10 seconds.' }; }
  if (sym !== current) return;
  schedulePoll(POLL_MS);
  if (d.error) {
    if (!lastUpdate) document.getElementById('main').innerHTML =
      `<div class="card state"><h2>No forecast for ${esc(sym)}</h2><span class="err">${esc(d.error)}</span><br>Check the symbol on finance.yahoo.com, or pick one from the list.</div>`;
    return;
  }
  lastUpdate = Date.now();
  render(d);
}

/* ---------- sections ---------- */
const marketState = d => d.market ? d.market.state : (d.data_age_sec < 600 ? 'open' : 'closed');

function statusChip(d) {
  const st = marketState(d);
  if (st === 'break') return `<span class="chip closed">Lunch break</span>`;
  if (st !== 'open') return `<span class="chip closed">Market closed</span>`;
  if (d.data_age_sec > 120) return `<span class="chip delayed">Open · prices ${Math.round(d.data_age_sec / 60)} min delayed</span>`;
  return `<span class="chip open">Open · live</span>`;
}

function forecastSection(d) {
  const f = d.forecast, fm = f.models;
  const vals = [f.last, ...Object.values(fm).flatMap(q => [q.q10, q.q90])];
  const lo = Math.min(...vals), hi = Math.max(...vals), pad = (hi - lo) * 0.12 || f.last * 1e-4;
  const a = lo - pad, b = hi + pad, x = v => ((v - a) / (b - a) * 100).toFixed(2) + '%';
  const rows = d.models.map(m => {
    const q = fm[m.key];
    if (!q) return `${mname(m)}<div class="fc-na">${m.available ? 'Not enough history for this ticker' : 'Not loaded. ' + esc(m.error || '')}</div>`;
    return `${mname(m)}<div class="fc-range num">${fmt(q.q10)} – ${fmt(q.q90)}</div>
      <div class="lane" title="${esc(m.name)}: 8 in 10 chance between ${fmt(q.q10)} and ${fmt(q.q90)}, median ${fmt(q.q50)}">
        <div class="track"></div>
        <div class="band" style="left:${x(q.q10)};width:${((q.q90 - q.q10) / (b - a) * 100).toFixed(2)}%;background:var(--s-${m.key})"></div>
        <div class="med" style="left:${x(q.q50)}"></div>
        <div class="now" style="left:${x(f.last)}"></div>
      </div>`;
  }).join('');
  return `<section class="card">
    <div class="id-row">
      <div><h1>${esc(d.name)}</h1>
        <div class="sub"><span class="num">${esc(d.symbol)}</span> · ${esc(d.exchange)} · ${statusChip(d)}</div></div>
      <div><div class="price num">${fmt(d.last_price)}${d.currency ? ' ' + esc(d.currency) : ''}</div>
        <div class="sub"><span style="color:${d.session_change >= 0 ? 'var(--good)' : 'var(--bad)'}">${signed(d.session_change * 100)}%</span> today · last price ${esc(d.last_bar)}</div></div>
    </div>
    <div class="fc">
      <div class="eyebrow">Where will the ${esc(f.target)} price land? · 8 in 10 chance it's inside each range</div>
      <div class="fc-rows">${rows}<div class="axis-row num"><span>${fmt(a)}</span><span>dashed line = now ${fmt(f.last)}</span><span>${fmt(b)}</span></div></div>
    </div>
  </section>`;
}

function livePanel(d) {
  const L = d.live, f = d.forecast, open = marketState(d) === 'open', r = L.last_result;
  if (r && lastLiveTarget && r.target !== lastLiveTarget) flashUntil = Date.now() + 2500;
  if (r) lastLiveTarget = r.target;
  const waitSub = !open ? 'Market is closed. Checks resume when trading starts.'
    : L.latency_sec > 120 ? `Yahoo publishes this exchange about ${Math.round(L.latency_sec / 60)} min late`
    : 'A minute\'s price is published shortly after it ends';
  const ms = d.models.filter(m => m.available);
  const streaks = ms.map(m => {
    const s = L.streak[m.key] || [], sc = L.scores[m.key] || {};
    const cells = s.map(x => `<span class="cell ${x.inside ? '' : 'miss'}" title="${esc(x.t)}: ${x.inside ? 'inside' : 'outside'}"></span>`).join('')
      + Array.from({ length: Math.max(0, 10 - s.length) }, () => '<span class="cell empty"></span>').join('');
    return `${mname(m)}<span class="cells" aria-hidden="true">${cells}</span>
      <span class="tally">${sc.n ? `<b style="color:var(--ink)">${Math.round(sc.coverage80 * sc.n)} of ${sc.n}</b> inside` : 'no results yet'}</span>`;
  }).join('');
  return `<section class="card" aria-live="polite">
    <div class="card-head"><div class="eyebrow"><span class="live-dot ${open ? '' : 'off'}"></span>Live check</div>
      <span class="updated" id="updated"></span></div>
    <div class="steps">
      <div class="step">
        <div class="step-title"><span class="step-n">1</span>Forecasts locked in</div>
        <div class="step-main num">${esc(L.made_at_local)}</div>
        <div class="step-sub">All ${ms.length} models saved their range before the ${esc(f.target)} price existed</div>
      </div>
      <div class="step ${open ? 'active' : ''}">
        <div class="step-title"><span class="step-n">2</span>Waiting for the ${esc(f.target)} price</div>
        <div class="step-main num" id="countdown">${open ? '…' : 'Paused'}</div>
        ${open ? '<div class="progress"><div id="progress" style="width:0%"></div></div>' : ''}
        <div class="step-sub">${waitSub}</div>
      </div>
      <div class="step ${Date.now() < flashUntil ? 'flash' : ''}">
        <div class="step-title"><span class="step-n">3</span>${r ? `Result for ${esc(r.target)}: <b class="num" style="color:var(--ink);margin-left:6px">${fmt(r.actual)}</b>` : 'Result'}</div>
        ${r ? `<div class="res">${ms.filter(m => r.models[m.key]).map(m => `<div>${mname(m)}
            <span class="chip ${r.models[m.key].inside ? 'inside' : 'outside'}">${r.models[m.key].inside ? '✓ Inside' : '✕ Outside'}</span></div>`).join('')}</div>`
          : `<div class="step-main" style="color:var(--muted)">Not yet</div>
             <div class="step-sub">The first result shows up when the ${esc(f.target)} price arrives</div>`}
      </div>
    </div>
    <div class="streaks">${streaks}</div>
  </section>`;
}

function covClass(v) { if (v == null) return ''; const e = Math.abs(v - .8); return e <= .05 ? 'cov-good' : e <= .12 ? 'cov-warn' : 'cov-bad'; }

function scoreboard(d) {
  const src = tab === 'live' ? d.live.scores : d.replay;
  const ms = d.models.filter(m => m.available && src[m.key] && src[m.key].n);
  const best = ms.length > 1 ? ms.reduce((a, b) => src[a.key].pinball_bps <= src[b.key].pinball_bps ? a : b).key : null;
  const n = ms.length ? Math.max(...ms.map(m => src[m.key].n)) : 0;
  const body = ms.length ? `<div class="table-wrap"><table>
      <tr><th>Model</th><th>Inside 8-in-10 range<small>target 80%</small></th><th>Range width<small>avg, bps</small></th>
        <th>Overall score<small>pinball loss, lower wins</small></th><th>Direction right<small>coin flip 50%</small></th></tr>
      ${ms.map(m => { const s = src[m.key]; return `<tr><td>${mname(m)}${m.key === best ? '<span class="best">Best</span>' : ''}</td>
        <td class="num ${covClass(s.coverage80)}">${pct(s.coverage80)}</td><td class="num">${s.width80_bps.toFixed(1)}</td>
        <td class="num">${s.pinball_bps.toFixed(3)}</td><td class="num">${pct(s.direction_acc)}</td></tr>`; }).join('')}
    </table></div>`
    : `<p class="note">${tab === 'live' ? 'Live results appear here as each minute is checked. Keep this tab open.' : 'Not enough history to replay yet.'}</p>`;
  return `<section class="card">
    <div class="card-head"><h2>Scoreboard${n ? ` · ${n} minutes` : ''}</h2>
      <div class="seg" role="group" aria-label="Score source">
        <button data-tab="replay" aria-pressed="${tab === 'replay'}">Replay · last 2 hours</button>
        <button data-tab="live" aria-pressed="${tab === 'live'}">Live · this session</button></div></div>
    ${body}
    <p class="note">${tab === 'replay' ? 'Replay runs every model over recent minutes, each forecast using only the prices before it.' : 'Live counts only forecasts saved before their price existed.'}
      The overall score rewards ranges that catch the price <i>and</i> stay narrow, so a model can't win by guessing wide.</p>
  </section>`;
}

function chartSection(d) {
  return `<section class="card">
    <div class="card-head"><h2>Forecast ranges vs. actual price, last hour</h2>
      <div class="toggles">${d.models.filter(m => m.available).map(m =>
        `<button class="toggle" data-model="${esc(m.key)}" aria-pressed="${!hidden.has(m.key)}">${sw(m.key)}${esc(m.name)}</button>`).join('')}</div></div>
    <div class="chart-box"><canvas id="chart" aria-label="Each model's 8-in-10 range per minute with the actual price"></canvas></div>
    <div class="legend-line"><span><span class="dot"></span>Actual price</span><span>Bars: each model's 8-in-10 range, forecast one minute ahead</span></div>
  </section>`;
}

function detailsSection(d) {
  const rep = d.report;
  const trainTable = rows => `<div class="table-wrap"><table><tr><th>Model</th><th>Inside 8-in-10</th><th>Width (bps)</th><th>Pinball (bps)</th><th>Direction</th></tr>
    ${rows.map(r => `<tr><td>${esc(r.model)}</td><td class="num">${pct(r['in 8-in-10 range'])}</td><td class="num">${(+r['avg range width (bps)']).toFixed(1)}</td>
      <td class="num">${(+r['pinball loss (bps)']).toFixed(3)}</td><td class="num">${pct(r['direction right'])}</td></tr>`).join('')}</table></div>`;
  const ms = d.models;
  return `<section class="card" style="padding-block:6px">
    <details id="d-train"><summary>Training results</summary>
      ${rep ? `<p>Trained ${esc(rep.trained_at)} on ${rep.tickers} tickers (${Number(rep.bars).toLocaleString()} one-minute prices).
        Chronos-2 was fine-tuned on the older weeks; every model's range width was then tuned on one held-out session.
        Scores below are from the last 2 sessions, which no model saw during training.</p>
        <p><b style="color:var(--ink)">Before training</b> (out of the box)</p>${trainTable(rep.zero_shot)}
        <p><b style="color:var(--ink)">After training</b></p>${trainTable(rep.trained)}`
      : '<p>No training report loaded. Run the training cells in the Colab notebook to fine-tune and score the models.</p>'}
    </details>
    <details id="d-log"><summary>Live forecast log</summary>
      ${d.live.log.length ? `<div class="table-wrap"><table><tr><th>Minute</th><th>Actual</th>${ms.filter(m => m.available).map(m => `<th>${esc(m.name)}</th>`).join('')}</tr>
        ${d.live.log.map(r => `<tr><td class="num">${esc(r.target)}</td><td class="num">${fmt(r.actual)}</td>${ms.filter(m => m.available).map(m => {
          const q = r.models[m.key]; if (!q) return '<td>–</td>';
          const ok = q[0] <= r.actual && r.actual <= q[1];
          return `<td class="num">${fmt(q[0])} – ${fmt(q[1])} ${r.gap ? '' : ok ? '<span class="chip inside">✓</span>' : '<span class="chip outside">✕</span>'}</td>`; }).join('')}</tr>`).join('')}
      </table></div>` : `<p>Each minute's forecasts are saved before the price exists, then checked here when it arrives.</p>`}
    </details>
    <details id="d-how"><summary>About the models</summary>
      ${ms.map(m => `<p>${mname(m)} ${esc(m.blurb || '')}${m.available ? ` · range width ×${(+m.scale).toFixed(2)} after tuning` : ' · not loaded'}</p>`).join('')}
      <p>All three predict the same thing: the 10th to 90th percentile of the next one-minute closing price. Whether a liquid stock goes up or down
      in the next minute is close to a coin flip for any model, so judge them on the range: it should catch about 80% of prices while staying as narrow as possible.</p>
    </details>
  </section>`;
}

/* ---------- render ---------- */
function render(d) {
  digits = d.last_price < 10 ? 4 : 2;
  const open = ['d-train', 'd-log', 'd-how'].filter(id => document.getElementById(id)?.open);
  if (chart) { chart.destroy(); chart = null; }
  document.getElementById('main').innerHTML =
    forecastSection(d) + livePanel(d) + scoreboard(d) + chartSection(d) + detailsSection(d);
  open.forEach(id => document.getElementById(id).open = true);
  document.querySelectorAll('.seg button').forEach(b => b.addEventListener('click', () => {
    tab = b.dataset.tab; try { localStorage.setItem('mfl-tab', tab); } catch {}
    render(d);
  }));
  document.querySelectorAll('.toggle').forEach(b => b.addEventListener('click', () => {
    const k = b.dataset.model; hidden.has(k) ? hidden.delete(k) : hidden.add(k);
    try { localStorage.setItem('mfl-hidden', JSON.stringify([...hidden])); } catch {}
    b.setAttribute('aria-pressed', String(!hidden.has(k))); drawChart(d);
  }));
  drawChart(d);
  startLiveClock(d);
}

function drawChart(d) {
  if (chart) { chart.destroy(); chart = null; }
  const pts = d.points, ms = d.models.filter(m => m.available && !hidden.has(m.key));
  const labels = pts.map(p => p.t);
  const vals = pts.flatMap(p => [p.actual, ...ms.flatMap(m => p.ranges[m.key] || [])]).filter(v => v != null);
  const span = Math.max(...vals) - Math.min(...vals) || vals[0] * 1e-4;
  const datasets = ms.map(m => ({
    type: 'bar', label: m.name, data: pts.map(p => p.ranges[m.key] || null), backgroundColor: color(m.key) + 'b3',
    borderRadius: 2, borderSkipped: false, barPercentage: .9, categoryPercentage: .85, order: 2,
  }));
  datasets.push({ type: 'line', label: 'Actual', data: pts.map(p => p.actual), showLine: false, order: 1,
    pointRadius: 3, pointHoverRadius: 6, pointBackgroundColor: tok('--ink'), pointBorderColor: tok('--panel'), pointBorderWidth: 1.5 });
  chart = new Chart(document.getElementById('chart'), {
    data: { labels, datasets },
    options: {
      responsive: true, maintainAspectRatio: false, animation: false,
      interaction: { mode: 'index', intersect: false },
      plugins: { legend: { display: false }, tooltip: { filter: i => i.raw != null, callbacks: {
        label: c => c.dataset.type === 'bar' ? `${c.dataset.label}: ${fmt(c.raw[0])} – ${fmt(c.raw[1])}` : `Actual: ${fmt(c.raw)}` } } },
      scales: {
        x: { ticks: { color: tok('--faint'), maxTicksLimit: 8, maxRotation: 0 }, grid: { display: false }, border: { color: tok('--line') } },
        y: { position: 'right', beginAtZero: false, suggestedMin: Math.min(...vals) - span * .06, suggestedMax: Math.max(...vals) + span * .06,
             ticks: { color: tok('--faint'), maxTicksLimit: 6, callback: v => fmt(v) }, grid: { color: tok('--line') }, border: { display: false } },
      },
    },
  });
}

function startLiveClock(d) {
  clearInterval(liveClock);
  const L = d.live, open = marketState(d) === 'open';
  const startMs = new Date(L.made_at).getTime(), dueMs = Date.now() + L.expected_in_sec * 1000;
  let fast = false;
  const tick = () => {
    const up = document.getElementById('updated');
    if (up && lastUpdate) up.textContent = `Checked ${Math.round((Date.now() - lastUpdate) / 1000)}s ago`;
    if (!open) return;
    const left = Math.round((dueMs - Date.now()) / 1000);
    const cd = document.getElementById('countdown'), pr = document.getElementById('progress');
    if (cd) cd.textContent = left > 0 ? `~${left >= 60 ? Math.floor(left / 60) + 'm ' : ''}${left % 60}s` : 'Arriving any moment…';
    if (pr) pr.style.width = Math.min(100, Math.max(2, (Date.now() - startMs) / Math.max(1, dueMs - startMs) * 100)) + '%';
    if (left <= 0 && !fast) { fast = true; schedulePoll(3000); }       // check more often once it's due
  };
  tick(); liveClock = setInterval(tick, 1000);
}

(async () => {
  const markets = await loadMarkets();
  let start = decodeURIComponent(location.hash.slice(1)) || null;
  try { start = start || localStorage.getItem('mfl-ticker'); } catch {}
  start = start || markets.find(m => m.state === 'open')?.tickers[0].symbol;
  if (start) select(start);
  setInterval(loadMarkets, 60000);
})();
</script>
</body>
</html>

In [ ]:
import sys, torch, numpy as np, pandas as pd
sys.path.insert(0, "/content/lab")
from riskmc import training as T
from riskmc.forecasters import build
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4 GPU for speed)")

## 2. Download training data
About 4 weeks of 1-minute prices per ticker (Yahoo Finance keeps 1-minute bars for ~30 days). Each ticker's history is split by time:

`[ fine-tune (older weeks) ][ calibrate: 1 session ][ test: last 2 sessions ]`

In [ ]:
DATA = T.download_all(T.TRAIN_TICKERS, days=28)
print(f"\n{len(DATA)} tickers, {sum(len(s) for s in DATA.values()):,} one-minute prices")

## 3. Load the models (out of the box)

In [ ]:
MODELS = build(device=DEVICE)
for m in MODELS.values():
    print(f"{m.name:<12} {'loaded' if m.available else 'FAILED: ' + str(m.error)}")

## 4. Score the untrained models on the test sessions
This is the baseline the training has to beat.

In [ ]:
N_TEST = 150   # forecasts per ticker
ZERO_SHOT = T.summarize(MODELS, T.collect(MODELS, DATA, "test", N_TEST))
ZERO_SHOT.style.format(precision=3)

## 5. Fine-tune Chronos-2
Uses chronos-forecasting's built-in `fit()` with LoRA adapters on the older weeks of every ticker.

In [ ]:
FT_SERIES = T.finetune_series(DATA)
print(len(FT_SERIES), "training series,", sum(len(s) for s in FT_SERIES), "prices")
if MODELS["chronos2"].available:
    MODELS["chronos2"].fine_tune(FT_SERIES, out_dir="/content/models/chronos2-lora", num_steps=600)

## 6. Fine-tune TimesFM 2.5
LoRA fine-tuning following TimesFM's own example: random 512-minute windows, predict the next minute,
minimizing the model's built-in quantile loss.

In [ ]:
if MODELS["timesfm"].available:
    LOSSES = MODELS["timesfm"].fine_tune(FT_SERIES, out_dir="/content/models/timesfm-lora", num_steps=600)
    pd.Series(LOSSES).rolling(25).mean().plot(title="TimesFM fine-tuning loss", figsize=(7, 3));

## 7. Tune every model's range width
One held-out session is used to fit a single spread multiplier per model (Monte Carlo included),
so all three get the same calibration treatment.

In [ ]:
for m in MODELS.values():
    m.scale = 1.0
SCALES = T.calibrate(MODELS, T.collect(MODELS, DATA, "calib", N_TEST))

## 8. Score the trained models on the test sessions

In [ ]:
TEST_ROWS = T.collect(MODELS, DATA, "test", N_TEST)
TRAINED = T.summarize(MODELS, TEST_ROWS)
print("Before training"); display(ZERO_SHOT.style.format(precision=3))
print("After training");  display(TRAINED.style.format(precision=3))

In [ ]:
import matplotlib.pyplot as plt
from riskmc.minute import score
by_market = (TEST_ROWS.assign(market=TEST_ROWS.symbol.map({s: m["name"] for m in T.MARKETS for s, _ in m["tickers"]}))
             .groupby(["market", "model"]).apply(lambda r: score(r)["pinball_bps"]).unstack())
by_market = by_market.rename(columns={k: m.name for k, m in MODELS.items()})
ax = by_market.plot.barh(figsize=(8, 7), color=["#eb6834", "#2a78d6", "#1baf7a"][:by_market.shape[1]])
ax.set_xlabel("Pinball loss, bps (lower is better)"); ax.set_title("Test sessions, by market"); plt.tight_layout()

In [ ]:
REPORT = T.save_report("/content/models/report.json", MODELS, ZERO_SHOT, TRAINED, SCALES, DATA,
                       extra={"device": DEVICE})
print("Saved /content/models/report.json")

## 9. Download the trained models
Saves `trained_models.zip` to your computer: both LoRA adapters (a few MB each) and the range widths.
Unzip it into the project's `models/` folder and run `python app.py` locally (see the README).

In [ ]:
import shutil
from pathlib import Path
pkg = Path("/content/trained_models")
shutil.rmtree(pkg, ignore_errors=True); pkg.mkdir()
shutil.copy("/content/models/report.json", pkg / "report.json")
if Path("/content/models/chronos2-lora/finetuned-ckpt").exists():
    shutil.copytree("/content/models/chronos2-lora/finetuned-ckpt", pkg / "chronos2-lora")
if Path("/content/models/timesfm-lora").exists():
    shutil.copytree("/content/models/timesfm-lora", pkg / "timesfm-lora")
shutil.make_archive("/content/trained_models", "zip", pkg)
!du -sh /content/trained_models.zip
from google.colab import files
files.download("/content/trained_models.zip")

## 10. Live comparison app
Opens the app in a new tab. Pick a ticker from a market that is open now; every minute all three models
lock in a forecast, and each is checked when the real price arrives.

In [ ]:
import app
app.serve(MODELS, REPORT, port=5050, background=True)
from google.colab import output
output.serve_kernel_port_as_window(5050, anchor_text="Open Minute Forecast Lab")

In [ ]:
# Or show it inside the notebook:
output.serve_kernel_port_as_iframe(5050, height=1400)